# VitalDB waveform ingestion

Downloads the selected waveform tracks, harmonizes sampling, verifies outputs, and supports restart. Complete cohort discovery first and enable full download only after the pilot passes.

Run the setup cell first. Full processing and training require locally provisioned datasets; the grading quick start uses synthetic tests and saved aggregate results.

Restart the kernel after changing the interpreter or installed dependencies, then rerun setup before resuming ingestion.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import sys
import time

_SETUP_STARTED = time.perf_counter()
candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next((p for p in candidates if (p / "notebooks" / "01-vitaldbaccess.ipynb").exists()), None)
if ROOT is None:
    raise RuntimeError("Open the project root or its notebooks folder")
EXPECTED_PYTHON = ROOT / ".venv" / "Scripts" / "python.exe"
assert Path(sys.executable).resolve() == EXPECTED_PYTHON.resolve(), "Select the project .venv kernel"
sys.path.insert(0, str(ROOT / "notebooks"))

MODE = "FULL_DOWNLOAD"                   # PILOT | FULL_DOWNLOAD | RESUME
RESUME_TARGET = "FULL_DOWNLOAD"          # PILOT | FULL_DOWNLOAD; never inferred silently
# Plain json only: numpy/pandas/vitaldb/scipy/hemodynamics load in the timed load step of the download cell.
SETTINGS = json.loads((ROOT / "configs" / "hemodynamics.json").read_text(encoding="utf-8"))
MIN_FREE_SPACE_BYTES = int(float(SETTINGS["min_free_space_gb"]) * 1_000_000_000)
TARGET_WAVEFORM_FS = float(SETTINGS["target_waveform_fs"])
MAX_ATTEMPTS = int(SETTINGS["max_retries"])
BACKOFF_SECONDS = float(SETTINGS["vitaldb"]["retry_backoff_seconds"])
print("Interpreter:", sys.executable)
print("Root:", ROOT)
print("Mode:", MODE, "resume target:", RESUME_TARGET)
print("Patient seed:", SETTINGS["patient_split_seed"], "target Hz:", TARGET_WAVEFORM_FS, "allow full:", SETTINGS["allow_full_vitaldb_download"])


def preflight(root=ROOT, sample_seconds=0.8):
    """About one second, psutil only. Warns about contention; never aborts."""
    import psutil
    started = time.perf_counter()
    me = psutil.Process()
    peers = []
    for proc in psutil.process_iter(["pid", "name"]):
        if proc.info["pid"] != me.pid and "python" in (proc.info["name"] or "").lower():
            try:
                proc.cpu_percent(None)
                peers.append(proc)
            except psutil.Error:
                pass
    time.sleep(sample_seconds)
    busy = []
    for proc in peers:
        try:
            cpu = proc.cpu_percent(None)
            if cpu > 50:
                command = " ".join(proc.cmdline()).lower()
                role = ("MOVER" if "mover" in command else "VitalDB" if "vitaldb" in command else "ipykernel" if "ipykernel" in command
                        else "pool worker" if "multiprocessing" in command else "python")
                busy.append((proc.pid, role, cpu))
        except psutil.Error:
            pass

    def lock_owner(path):
        if not path.exists():
            return None
        try:
            data = json.loads(path.read_text(encoding="utf-8"))
            pid = int(data["pid"]) if isinstance(data, dict) else int(data)
            created = data.get("create_time") if isinstance(data, dict) else None
        except (OSError, ValueError, KeyError, TypeError):
            return {"pid": None, "alive": None, "text": "lock file present, owner unknown"}
        alive = psutil.pid_exists(pid)
        if alive and created:
            try:
                alive = abs(psutil.Process(pid).create_time() - float(created)) < 2
            except psutil.Error:
                alive = False
        return {"pid": pid, "alive": alive, "text": f"pid {pid} " + ("running" if alive else "not running (stale lock)"),
                "mode": data.get("mode") if isinstance(data, dict) else None}

    mover = lock_owner(root / "data" / "checkpoints" / "mover" / "fast_ingest.lock")
    vitaldb = lock_owner(root / "data" / "checkpoints" / "vitaldb" / "run.lock")
    free_disk = shutil.disk_usage(root).free
    memory = psutil.virtual_memory()
    print(f"Preflight: kernel PID {me.pid}, up {time.time() - me.create_time():.0f}s | free disk {free_disk / 1e9:.1f} GB "
          f"(reserve {MIN_FREE_SPACE_BYTES / 1e9:.0f} GB) | free RAM {memory.available / 1e9:.1f} of {memory.total / 1e9:.0f} GB")
    shown = ", ".join(f"{role} pid {pid} {cpu:.0f}%" for pid, role, cpu in sorted(busy, key=lambda item: -item[2])[:5])
    print(f"Preflight: other python processes above 50% CPU (1 s sample): {len(busy)}" + (f" ({shown})" if busy else ""))
    print(f"Preflight: MOVER lock: {mover['text'] if mover else 'none'} | VitalDB run lock: {vitaldb['text'] if vitaldb else 'none'}")
    warnings = []
    if (mover and mover["alive"] is not False) or any(role == "MOVER" for _, role, _ in busy):
        warnings.append("a MOVER pipeline appears to be running")
    if vitaldb and vitaldb["alive"]:
        warnings.append(f"another VitalDB run is active ({vitaldb['text']}, {vitaldb.get('mode')})")
    if len(busy) >= 4:
        warnings.append(f"{len(busy)} python processes are saturating the CPU")
    if free_disk < MIN_FREE_SPACE_BYTES:
        warnings.append("free disk is already below the reserve; the download will stop immediately")
    for text in warnings:
        print(f"*** WARNING: {text}. Imports and downloads will be slow or conflict; wait, or Interrupt/restart the other run first. ***")
    print(f"Preflight took {time.perf_counter() - started:.1f}s")


preflight()
print(f"Setup finished in {time.perf_counter() - _SETUP_STARTED:.1f}s (heavy imports are deferred to the download cell).")


## 1. Run the download
**Order.** A FULL run dispatches unfinished cases in a seeded (`patient_split_seed` 5052) representative order: **any prefix is about 70/15/15 by split** and the first surgery of every patient comes before any repeat surgery, so a partial run is already a usable sample. Verified cases (hash/readback) are skipped and never re-downloaded; `PILOT` and explicit `case_ids` keep sorted order.

**Speed (measured, 8 workers on this machine).** About 1.1 to 1.5 s per case, roughly 10 MB read from the network and 28 MB written per case, so the remaining cases take on the order of one hour (an extrapolation, not a promise). Throughput is network-bound; the RAM guard (`max_inflight_ram_gb`) and the 100 GB disk reserve stay active.

**Stop and resume.** Press **Interrupt once**: the run drains briefly, saves committed cases and returns `PAUSED`. Rerun the **START HERE** cell (or set `MODE = "RESUME"` with the same `RESUME_TARGET`); only unverified cases are fetched. Starting a run re-verifies recorded cases at roughly 30 cases/s, and the start-up lines are time-stamped so a stall shows where.

**Result states.** `COMPLETE`: all 3,185 cases verified. `PAUSED`: interrupted, disk reserve or RAM guard stop; rerun to continue. `INCOMPLETE`: the run ended (or was capped by `MAX_CASES`) with cases still unverified, including failed ones; rerun to retry them. `FAILED`: failures and nothing completed.

**Logs and reports.** One JSON line per finished attempt: `logs/vitaldb_download_<UTC timestamp>.jsonl` (seconds per phase, bytes, attempts, status). The end-of-run report, also written after `PAUSED` runs: `results/vitaldb_download_report.json`. Output manifest: `data/manifests/vitaldb_download_manifest.csv`/`.parquet`. When a FULL run finishes `COMPLETE`, the ready-window index and unified manifest (section 5) are built automatically; a failure there only prints a warning.


In [ ]:
# RUN VITALDB DOWNLOAD - START HERE
# MODE and RESUME_TARGET are configured in the setup cell above.
# Full run: MODE="FULL_DOWNLOAD". Restart after Interrupt: rerun this cell (or MODE="RESUME", RESUME_TARGET="FULL_DOWNLOAD").
# MAX_CASES=None means the entire selected mode; an integer is a bounded diagnostic batch (still representative-ordered in FULL mode).
if "ROOT" not in globals():
    raise RuntimeError("Run the setup cell above first")

# Explicit, timed load step: the slow imports live here, not in the setup cell.
import importlib
_load_started = time.perf_counter()
_reload = "hemodynamics.vitaldb_pipeline" in sys.modules
import numpy as np
import pandas as pd
from hemodynamics import canonical, vitaldb_pipeline
if _reload:
    vitaldb_pipeline = importlib.reload(vitaldb_pipeline)
run_pipeline = vitaldb_pipeline.run_pipeline
audit_project = vitaldb_pipeline.audit_project
assert_originals_unchanged = vitaldb_pipeline.assert_originals_unchanged
load_settings = vitaldb_pipeline.load_settings
print(f"Library load (numpy, pandas, vitaldb, scipy, hemodynamics) took {time.perf_counter() - _load_started:.1f}s", flush=True)

SETTINGS = load_settings(ROOT)
MAX_CASES = None  # Explicit full-mode default; set a number only for a bounded diagnostic.
DOWNLOAD_WORKERS = int(SETTINGS["vitaldb"].get("download_workers", 8))
TARGET_WAVEFORM_FS = float(SETTINGS["target_waveform_fs"])
cohort = pd.read_csv(ROOT / "data" / "manifests" / "usable_cohort.csv")
recorded = sum(row["status"] == "complete" for row in canonical.CheckpointStore(ROOT, "vitaldb").rows())
print("RUN VITALDB DOWNLOAD - START HERE", flush=True)
print(f"Mode={MODE}, resume_target={RESUME_TARGET}, max_cases={MAX_CASES}", flush=True)
print(f"Workers={DOWNLOAD_WORKERS} (hard max {vitaldb_pipeline.MAX_DOWNLOAD_WORKERS}), RAM budget={SETTINGS['vitaldb'].get('max_inflight_ram_gb', 8)}GB, math threads=1", flush=True)
print(f"Free disk={shutil.disk_usage(ROOT).free / 1e9:.2f}GB; reserve={SETTINGS['min_free_space_gb']} decimalGB", flush=True)
print(f"allow_full_vitaldb_download={SETTINGS['allow_full_vitaldb_download']}; recorded complete={recorded}/{len(cohort)} (re-verified by hash/readback and skipped when the run starts)", flush=True)
print("Interrupt the kernel ONCE to pause safely; rerun this cell to resume. Live line: verified/total, cases/min, ETA, network MB/s, GB written, phases, retries, last commit.", flush=True)
result = run_pipeline(ROOT, mode=MODE, resume_target=RESUME_TARGET,
                      download_workers=DOWNLOAD_WORKERS, max_cases=MAX_CASES, progress=True)
audit = json.loads((ROOT / "data" / "manifests" / "vitaldb_project_audit.json").read_text(encoding="utf-8"))
if result["summary_status"] == "PAUSED":
    print("Paused safely: rerun this cell (or MODE=RESUME with the same RESUME_TARGET).")
if result["failed"]:
    print("Failures are checkpointed for RESUME:", result["failed"])


## 2. Progress, ETA and report
The live line printed during the run shows `verified/total`, `cases/min`, `ETA`, `net_MB/s` (compressed bytes actually read from the network), `written_GB`, in-flight counts per phase (network/resample/write/readback), `retries`, `failed` and the last durable commit time. This cell summarizes the finished or paused run from the persisted report; its ETA is an extrapolation from this run's wall seconds per case.


In [ ]:
from datetime import timedelta
report_path = ROOT / "results" / "vitaldb_download_report.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
eta = report["eta_remaining_sec_extrapolated"]
display(pd.Series({
    "status": report["summary_status"], "verified": f"{report['verified_cases']}/{report['cohort_cases']}",
    "remaining": report["remaining_cases"], "completed this run": report["completed_this_run"],
    "skipped (already verified)": report["skipped_verified"], "failed": report["failed_cases"],
    "workers": report["download_workers"], "retries": report["retries"], "HTTP 429": report["http_429"],
    "seconds/case (wall)": None if report["seconds_per_case_wall"] is None else round(report["seconds_per_case_wall"], 2),
    "cases/min": None if report["cases_per_min"] is None else round(report["cases_per_min"], 1),
    "ETA (extrapolated)": None if eta is None else str(timedelta(seconds=int(eta))),
    "network MB/s": None if report["network_MB_per_sec"] is None else round(report["network_MB_per_sec"], 2),
    "GB written this run": round(report["written_bytes_this_run"] / 1e9, 3), "total output GB": round(report["total_output_GB"], 3),
    "seconds by phase (all attempts)": report["phase_seconds_total"], "attempt log": report["attempt_log"],
}, name="last run").to_frame())
if report["failures"]:
    display(pd.Series(report["failures"], name="failure").to_frame())
print("Report:", report_path)


### Method notes (unchanged)
Native 500 Hz records share a common native clock (at most 1 ms timestamp quantization). **`scipy.signal.resample_poly` applies an anti-alias low-pass FIR before 100 Hz decimation**. Contiguous finite samples merge across packet boundaries; NaNs/time gaps split independent filtering runs. Filter phase is anchored to the recording-wide grid. Reflect padding stays internal to each finite run; an exact observed-span mask keeps missing waveform slots NaN. No gap interpolation or fill.

MAP retains native timestamps/values, with no 1 Hz upsampling or forward filling. Canonical atomic writes, hashes and readback must pass before completion. Shared `configs/hemodynamics.json` controls seeds 5052, target Hz, disk reserve, retry settings, VitalDB-specific thresholds and filter settings. Processing signatures reject old unfiltered/differently seeded outputs instead of skipping them.

`FULL_DOWNLOAD` requires **both** explicit `allow_full_vitaldb_download=true` and all three pilot readbacks plus a no-download resume; the pilot gate is already satisfied for this project. Set the flag to `false` to block a full run.


## 3. Revalidate immutable source evidence, pilot restart proof and manifest
Original cohort thresholds: duration >= 30 minutes, every waveform missingness **strictly <20%**, >= 300 non-NaN MAP observations, >= 10 MAP observations/minute, coverage **span** >= 80%, median MAP gap <= 10 seconds. These are provisional data-engineering checks, not physiological SQI. All exact track names are verified against the local reference dictionary and public track index. Both ART_MBP device aliases exist; explicit Solar8000 preference is recorded.

Detailed reproducible facts and original SHA256 hashes: `data/manifests/vitaldb_project_audit.md` and `.json`.


In [ ]:
audit = audit_project(ROOT)
print(json.dumps({key: audit[key] for key in (
    "final_usable_cases", "usable_patients", "patient_id_documentation",
    "quality_criteria", "cohort_map_source_counts", "pilot_case_ids", "patient_splits"
)}, indent=2))
assert all(value == 0 for value in audit["patient_split_intersections"].values())
splits = pd.read_parquet(ROOT / "data" / "manifests" / "vitaldb_patient_splits.parquet")
assert splits.groupby("patient_id").split.nunique().eq(1).all()
display(splits.head())


In [ ]:
# Pilot-only verified restart check: never triggers an additional full run.
if result["target"] == "PILOT" and result["summary_status"] != "PAUSED" and not result["failed"]:
    store = canonical.CheckpointStore(ROOT, "vitaldb")
    before_attempts = {case: store.get(case)["attempts"] for case in result["selected_cases"]}
    resume_result = run_pipeline(ROOT, mode="RESUME", resume_target="PILOT", progress=True,
                                 download_workers=DOWNLOAD_WORKERS, case_ids=result["selected_cases"])
    assert resume_result["processed"] == [] and resume_result["failed"] == []
    assert set(resume_result["skipped"]) == set(result["selected_cases"])
    assert before_attempts == {case: store.get(case)["attempts"] for case in result["selected_cases"]}
    print("Pilot outputs reused without downloads; this is not full-cohort completion.")
else:
    print("No automatic second run. To restart explicitly, set MODE=RESUME and RESUME_TARGET=FULL_DOWNLOAD in setup, then execute START HERE.")


In [ ]:
store = canonical.CheckpointStore(ROOT, "vitaldb")
rows = pd.DataFrame(store.rows())
columns = ["case_id", "patient_id", "split", "status", "attempts", "waveform_rows", "map_rows",
           "duration_sec", "map_median_gap_sec", "map_max_gap_sec", "processed_bytes", "readback_verified"]
display(rows.groupby(["split", "status"]).size().unstack(fill_value=0))
display(rows[[column for column in columns if column in rows]].tail(10))
print("Actual processed GB:", rows.loc[rows.status.eq("complete"), "processed_bytes"].sum() / 1e9)
print("Output manifest: data/manifests/vitaldb_download_manifest.csv / .parquet")


## 4. Canonical loader smoke test and real signal preview
Validate each pilot's time arrays and all four actual modalities. MAP gaps are reported, not imputed. Data remain entirely local; only public VitalDB downloads are performed.


In [ ]:
for case in audit["pilot_case_ids"]:
    loaded = canonical.load_case(ROOT, "vitaldb", case)
    assert loaded["dataset"] == "vitaldb"
    assert np.allclose(np.diff(loaded["waveform_time_sec"]), 1 / TARGET_WAVEFORM_FS, atol=1e-8)
    assert np.all(np.diff(loaded["map_time_sec"]) > 0)
    assert all(np.isfinite(loaded[signal]).any() for signal in ("ART", "ECG", "PPG", "MAP"))
    assert loaded["patient_id"] is not None and loaded["split"] in ("train", "validation", "test")
    assert loaded["metadata"]["patient_split_seed"] == SETTINGS["patient_split_seed"]
    assert loaded["metadata"]["resampling"] == "polyphase_contiguous_v1"
    assert loaded["metadata"]["time_reference"] == "seconds_since_recording_start"
    assert all(loaded["metadata"][f"{signal}_final_fs"] == TARGET_WAVEFORM_FS for signal in ("art", "ecg", "ppg"))
    print(case, "waveform rows", len(loaded["ART"]), "MAP rows", len(loaded["MAP"]),
          "native median MAP gap", np.median(np.diff(loaded["map_time_sec"])))
    del loaded
assert_originals_unchanged(ROOT)
print("All protected original files unchanged.")


In [ ]:
import matplotlib.pyplot as plt
case = audit["pilot_case_ids"][0]
loaded = canonical.load_case(ROOT, "vitaldb", case)
common = np.isfinite(loaded["ART"]) & np.isfinite(loaded["ECG"]) & np.isfinite(loaded["PPG"])
start = loaded["waveform_time_sec"][np.flatnonzero(common)[0]]
segment = (loaded["waveform_time_sec"] >= start) & (loaded["waveform_time_sec"] < start + 5)
fig, axes = plt.subplots(3, 1, figsize=(12, 6), sharex=True)
for axis, signal in zip(axes, ("ART", "ECG", "PPG")):
    axis.plot(loaded["waveform_time_sec"][segment], loaded[signal][segment], linewidth=0.8)
    axis.set_ylabel(signal)
axes[-1].set_xlabel("Seconds since recording start")
fig.suptitle(f"Actual VitalDB case {case}: five observed seconds (not synthesized)")
fig.tight_layout()
plt.show()
del loaded


## Notes: gates, resume and independence
- Keep `MODE="FULL_DOWNLOAD"` for the whole cohort, or `MODE="PILOT"` / `MODE="RESUME"; RESUME_TARGET="PILOT"` for the three-case repeat. Both gates (`allow_full_vitaldb_download=true` and the verified pilot) are checked; nothing starts automatically.
- Both seeds, target Hz, disk reserve, retries/backoff and VitalDB tracks/quality/filter settings come from central configuration. Changing scientific settings invalidates processing signatures and requires reprocessing before resume skips.
- Repeated subjects share their documented patient split; blank original surgery splits are not used for independence claims.
- Disk stop: checkpoints persist; incomplete files are not skipped. Original cohorts, reference data and MOVER archives remain untouched.


## 5. Build ready-window index and unified manifest
Run after the download is `COMPLETE`. A complete FULL run already triggers these two steps automatically (a failure there only prints a warning), so use this cell to (re)build them by hand, for example after a `RESUME`.


In [ ]:
# 5. Build ready-window index and unified manifest
from hemodynamics.unified import index_ready_windows, build_unified_index
index_ready_windows(ROOT, "vitaldb", progress=True)
display(build_unified_index(ROOT, progress=True).groupby(["dataset", "split_within_dataset"]).size())
